# Обработка аномалий при пост-анализе пилотов

Расчётный пример к [отчёту](report.md). Сравниваем средние по всем клиентам, после исключения выброса и после ограничения его значения. Все данные в примере синтетические.

In [1]:
import numpy as np
from sklearn.ensemble import IsolationForest

test = np.array([10.] * 99 + [1010.])
control = np.full(100, 20.)
values = np.concatenate([test, control])
is_test = np.arange(values.size) < test.size
print(f"Средние до обработки: тест {test.mean():.1f}, контроль {control.mean():.1f}.")

Средние до обработки: тест 20.0, контроль 20.0.


## Два правила отбора

Фиксированная граница 20 оставляет все значения, которые её не превышают. Во втором правиле Isolation Forest обучается на обеих группах без признака группы. Для этого примера берём границу между двумя наиболее аномальными различными оценками. Чем меньше значение `score_samples`, тем аномальнее наблюдение. Выбранный здесь порог иллюстрирует расчёт на этих данных.

In [2]:
forest = IsolationForest(n_estimators=500, max_samples=int(np.sqrt(values.size)), random_state=42, n_jobs=1)
scores = forest.fit(values[:, None]).score_samples(values[:, None])
distinct_scores = np.unique(scores)
score_threshold = distinct_scores[:2].mean()
outliers = {
    "Фиксированный порог": values > 20,
    "Isolation Forest": scores < score_threshold,
}
for rule, flagged in outliers.items():
    print(f"{rule}: отмечено {flagged.sum()}, значения {values[flagged].tolist()}")
    assert values[flagged].tolist() == [1010.]

Фиксированный порог: отмечено 1, значения [1010.0]
Isolation Forest: отмечено 1, значения [1010.0]


## Пересчёт средних

Для винсоризации используем границы оставшихся значений на объединённой выборке. Разницу считаем как среднее в тесте минус среднее в контроле.

In [3]:
for rule, flagged in outliers.items():
    keep = ~flagged
    capped = np.clip(values, values[keep].min(), values[keep].max())
    groups = [
        ("Все клиенты", test, control),
        ("Фильтрация", values[is_test & keep], values[~is_test & keep]),
        ("Винсоризация", capped[is_test], capped[~is_test]),
    ]
    print(rule)
    differences = []
    for method, test_values, control_values in groups:
        test_mean, control_mean = test_values.mean(), control_values.mean()
        difference = test_mean - control_mean
        differences.append(difference)
        print(f"  {method}: тест {test_mean:.1f}, контроль {control_mean:.1f}, разница {difference:.1f}")
    np.testing.assert_allclose(differences, [0, -10, -9.9])

Фиксированный порог
  Все клиенты: тест 20.0, контроль 20.0, разница 0.0
  Фильтрация: тест 10.0, контроль 20.0, разница -10.0
  Винсоризация: тест 10.1, контроль 20.0, разница -9.9
Isolation Forest
  Все клиенты: тест 20.0, контроль 20.0, разница 0.0
  Фильтрация: тест 10.0, контроль 20.0, разница -10.0
  Винсоризация: тест 10.1, контроль 20.0, разница -9.9


В этом примере обработка одного наблюдения даёт ненулевую разницу при равных исходных средних. Наблюдение о стабилизации средних при последовательном отсечении получено отдельно в пилотах и описано в отчёте.